# Phase 11 — Real Email Notification

Connects the existing, already-tested `EmailSender` abstraction to a REAL
delivery provider, so genuinely new/updated DGCA publications generate a
real email instead of only a local `.html` preview.

**No pipeline logic is duplicated or redesigned.** This notebook documents
and tests the existing architecture plus the small, targeted additions
Phase 11 needed:

1. Existing email architecture (inspected, not assumed)
2. Provider selected + why
3. Configuration
4. Email generation (preview mode)
5. New publication -> one email, duplicate prevention
6. Updated publication -> one updated email, duplicate prevention
7. Email provider failure -> scheduler continues
8. Malformed publication -> isolated failure
9. HTML validation
10. Direct PDF link validation -- no invented URLs
11. Explicit (non-automatic) email retry
12. Final end-to-end demonstration
13. Security audit
14. Real delivery test
15. Summary / final report


In [ ]:
import sys, json, glob, os
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import storage, orchestrator, scheduler, documents, comparison, analysis, email_pipeline
from pipeline.config import load_config, PipelineConfig
from pipeline.utils import configure_logging
import pymupdf

ENV_PATH = PROJECT_ROOT / "config" / ".env"
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
PREVIEW_DIR = PROJECT_ROOT / "data" / "processed" / "email_previews"

configure_logging(PROJECT_ROOT / "logs", "phase11_real_email_notification")
config = load_config(ENV_PATH)
storage.init_db()

print("Imports OK -- pipeline package loaded (including Phase 11's email_pipeline.py / orchestrator.py / scheduler.py additions).")


## 1. Existing email architecture (inspected before any changes)

Per the Phase 11 brief, this section reports what was found by inspection,
not assumption.


In [ ]:
print("=== EmailSender abstraction (pipeline/email_pipeline.py) ===")
print(f"Abstract base: {email_pipeline.EmailSender.__name__} -- is_configured() + send(subject, html_body, recipient) -> EmailSendResult")
print("Implementations found:")
for cls in (email_pipeline.PreviewEmailSender, email_pipeline.SMTPEmailSender, email_pipeline.MicrosoftGraphEmailSender):
    print(f"  - {cls.provider_name!r}: {cls.__name__}")

print()
print("=== DGCAUpdate fields (pipeline/email_pipeline.py) ===")
print(list(email_pipeline.DGCAUpdate.model_fields.keys()))

print()
print("=== Configuration variables in use (pipeline/config.py) -- names only, no values ===")
cfg_fields = [f for f in config.__dataclass_fields__.keys()]
print(cfg_fields)

print()
print("=== Duplicate-email prevention mechanism ===")
print("storage.ingest_publication() classifies by content_hash (title+category+published_date+pdf_url).")
print("process_single_publication() returns status='skipped_unchanged' on IngestOutcome.UNCHANGED,")
print("BEFORE the email stage ever runs -- verified live across Phases 7 and 10.")

print()
print("=== How run_dgca_pipeline() invokes email ===")
print("process_single_publication(): AI analysis -> build_dgca_update() -> render_email_html() ->")
print("validate_email_html() -> get_email_sender() -> .send() -> mark_email_sent()/mark_failed().")

print()
print("=== How the scheduler invokes the pipeline (unchanged from Phase 10) ===")
print("run_scheduler() -> run_once() -> run_dgca_pipeline() -> process_single_publication() per record.")


## 2. Provider decision: SMTP

Two real providers already exist in the codebase:

- **Microsoft Graph** (`MicrosoftGraphEmailSender`) -- fully implemented, but
  requires a full Azure AD app registration (`AZURE_CLIENT_ID`/`TENANT_ID`/
  `CLIENT_SECRET`), the application-level `Mail.Send` permission, and admin
  consent. None of that exists yet (`config/.env` still has placeholder
  Azure values).
- **SMTP** (`SMTPEmailSender`) -- fully implemented, generic `smtplib`, needs
  only an SMTP host/port/credentials. It can point at **any** legitimate
  SMTP account (e.g. an app-password-authenticated Gmail account, or
  Office 365's SMTP AUTH endpoint if enabled for the mailbox) and deliver to
  **any** recipient address, including a real Outlook/Microsoft 365 mailbox
  -- the recipient does not need to be the same account as the sender.

**Decision: SMTP.** It needs no Azure app registration or admin consent,
is already fully implemented and tested (Phase 1 validated it against a
local SMTP catcher), and is sufficient to deliver to the target Outlook
mailbox. Per the brief's own instruction ("do NOT require Azure if the
existing SMTP implementation can legitimately solve the requirement"),
Microsoft Graph is left as the already-implemented-but-inert future option
it was designed to be.


## 3. Configuration

`config/.env` currently has `EMAIL_PROVIDER=preview` and `TEST_MODE=true`
(unchanged, safe default) and a leftover local SMTP catcher
(`SMTP_HOST=localhost:1025`, no TLS) from Phase 1's SMTP test -- not a real
mailbox. Real delivery needs `EMAIL_PROVIDER=smtp`, `TEST_MODE=false`, and
real `SMTP_HOST`/`PORT`/`USERNAME`/`PASSWORD`/`USE_TLS` +
`OUTLOOK_SENDER_EMAIL`/`OUTLOOK_RECIPIENT_EMAIL` values in `config/.env`
(never in this notebook, never printed, never committed --
the pipeline documentation lists the variable names for
SMTP delivery).

### TEST 2 -- real provider configuration check

Reports presence only, never values.


In [ ]:
status = scheduler.check_email_configuration(config)
print("=== Email delivery configuration (SET/MISSING only -- no values shown) ===")
print(f"EMAIL_PROVIDER          : {status['email_provider']}")
print(f"TEST_MODE                : {status['test_mode']}")
print(f"SMTP configuration       : {'SET' if status['smtp_configured'] else 'MISSING'}")
print(f"Microsoft Graph config   : {'SET' if status['graph_configured'] else 'MISSING'}")
print(f"Sender (OUTLOOK_SENDER_EMAIL)      : {'SET' if status['sender_set'] else 'MISSING'}")
print(f"Recipient (OUTLOOK_RECIPIENT_EMAIL): {'SET' if status['recipient_set'] else 'MISSING'}")

real_delivery_ready = (
    status["email_provider"] == "smtp" and not status["test_mode"]
    and status["smtp_configured"] and status["recipient_set"]
)
print()
print(f"Ready for real delivery right now: {real_delivery_ready}")
print("TEST 2 PASSED -- configuration presence reported without exposing any secret value.")


## 4. Email generation + preview mode

### TEST 1 -- Preview

A synthetic NEW-publication email: HTML generated, validated, and confirmed
to never touch the network (the `preview` provider only ever writes a local
file).


In [ ]:
def make_pdf_bytes(text: str) -> bytes:
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 550, 750), text, fontsize=11)
    b = doc.tobytes()
    doc.close()
    return b


preview_files_before = set(glob.glob(str(PREVIEW_DIR / "*.html")))

demo_update = email_pipeline.DGCAUpdate(
    title="Phase 11 Preview Demonstration Publication",
    category="Guidance",
    published_date="2026-08-19",
    summary="This is a synthetic summary used only to validate email generation.",
    key_changes=["Synthetic key change for validation."],
    impact_level="MEDIUM",
    impacted_departments=["Flight Operations"],
    recommended_actions=["Review this notice."],
    dgca_url="https://www.dgca.gov.in/digigov-portal/example",
    pdf_url="https://www.dgca.gov.in/digigov-portal/example.pdf",
)
html_preview = email_pipeline.render_email_html(demo_update, demo_update.summary, test_mode=True)
validation = email_pipeline.validate_email_html(html_preview, demo_update, demo_update.summary)
print(f"HTML generated: {len(html_preview)} characters")
print(f"Validation passed: {validation.is_valid} (missing: {validation.missing})")

preview_config = PipelineConfig(email_provider="preview", test_mode=True)
sender = email_pipeline.get_email_sender(preview_config, PREVIEW_DIR)
send_result = sender.send("[DGCA][NEW][GUIDANCE] New Guidance Published — " + demo_update.title, html_preview, "recipient@example.com")
print(f"Send result: success={send_result.success} provider={send_result.provider}")

preview_files_after = set(glob.glob(str(PREVIEW_DIR / "*.html")))
new_files = preview_files_after - preview_files_before
assert validation.is_valid
assert send_result.success and send_result.provider == "preview"
assert len(new_files) == 1, "Exactly one local preview file should have been written -- no network call made."
print()
print("TEST 1 PASSED -- HTML generated, validation passed, no email sent (local preview file only).")


## 5. New publication -> one email, duplicate prevention

Uses `process_single_publication()` directly (the exact function
`run_dgca_pipeline()`/the scheduler call) against the real database, with a
synthetic, clearly-prefixed (`e2e-test-p11-*`) fixture so this is
repeatable without depending on live DGCA state. `EMAIL_PROVIDER` stays
`preview` for this test (real credentials aren't configured yet, and this
section is about duplicate-prevention/content logic, not transport) --
Section 9 exercises a real SMTP transport failure specifically.


In [ ]:
conn = storage.get_connection()

# Clean up any leftover Phase 11 fixtures from a previous run, and their
# preview files, so this notebook is deterministic and repeatable.
conn.execute("DELETE FROM publications WHERE publication_id LIKE 'e2e-test-p11-%'")
conn.commit()
for f in glob.glob(str(PREVIEW_DIR / "*e2e-test-p11*.html")):
    os.remove(f)
for f in DOCUMENTS_DIR.glob("e2e-test-p11-*"):
    f.unlink()

test_config = PipelineConfig(email_provider="preview", test_mode=True, llm_provider="fallback")

pub_new = {
    "source_id": "e2e-test-p11-new",
    "title": "Phase 11 Synthetic New Publication",
    "category": "Guidance",
    "published_date": "2026-08-19",
    "publication_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-new",
    "pdf_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-new-v1.pdf",
}
old_text = "This publication establishes baseline requirements for flight crew rest periods. Operators must comply within ninety days."

preview_files_before = set(glob.glob(str(PREVIEW_DIR / "*.html")))
r_new = orchestrator.process_single_publication(
    conn, pub_new, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(old_text),
)
preview_files_after_new = set(glob.glob(str(PREVIEW_DIR / "*.html")))
new_email_files = preview_files_after_new - preview_files_before

print(f"[new] outcome={r_new.outcome} status={r_new.status} email_success={r_new.email_success}")
assert r_new.outcome == "new" and r_new.status == "success" and r_new.email_success
assert len(new_email_files) == 1, f"Expected exactly ONE new email, found {len(new_email_files)}."
new_email_path = list(new_email_files)[0]
new_email_html = Path(new_email_path).read_text(encoding="utf-8")
assert "UPDATED PUBLICATION" not in new_email_html, "A first-time publication must not show the UPDATED comparison block."
print("New-publication email subject prefix present:", "dgca-new" in Path(new_email_path).name)
print()
print("TEST PASSED -- new publication produced exactly ONE email, correctly rendered as a NEW notification.")

# Same publication, unchanged content, processed again -> NO second email.
r_dup = orchestrator.process_single_publication(
    conn, pub_new, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(old_text),
)
preview_files_after_dup = set(glob.glob(str(PREVIEW_DIR / "*.html")))
print(f"[re-process, unchanged] outcome={r_dup.outcome} status={r_dup.status}")
assert r_dup.outcome == "unchanged" and r_dup.status == "skipped_unchanged"
assert preview_files_after_dup == preview_files_after_new, "No new email file should have been written."
print("TEST PASSED -- same unchanged publication processed again produced NO duplicate email.")


## 6. Updated publication -> one updated email, duplicate prevention

DGCA republishes with a new `pdf_url` (this is what changes the
`content_hash` and triggers `IngestOutcome.UPDATED` -- see
`storage.compute_content_hash()`). The email must clearly show it as an
UPDATE (dynamic subject + Previous/Current/Added/Removed/Modified/AI
Explanation block), and processing the same updated content again a second
time must not send a second email.


In [ ]:
new_text = old_text + " AMENDMENT: operators must additionally submit a compliance report within thirty days of this notice."
pub_updated = dict(pub_new)
pub_updated["pdf_url"] = "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-new-v2.pdf"

preview_files_before_upd = set(glob.glob(str(PREVIEW_DIR / "*.html")))
r_upd = orchestrator.process_single_publication(
    conn, pub_updated, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(new_text),
)
preview_files_after_upd = set(glob.glob(str(PREVIEW_DIR / "*.html")))
updated_email_files = preview_files_after_upd - preview_files_before_upd

print(f"[updated] outcome={r_upd.outcome} status={r_upd.status} comparison_status={r_upd.comparison_status!r}")
assert r_upd.outcome == "updated" and r_upd.status == "success" and r_upd.email_success
assert len(updated_email_files) == 1, f"Expected exactly ONE updated-publication email, found {len(updated_email_files)}."
updated_email_path = list(updated_email_files)[0]
updated_email_html = Path(updated_email_path).read_text(encoding="utf-8")
assert "UPDATED PUBLICATION" in updated_email_html, "The comparison block must be present for a genuine update."
assert "compliance report within thirty days" in updated_email_html, "The actual added text must appear (AI explanation / added-changes list)."
print("Updated-publication email filename indicates UPDATED subject prefix:", "dgca-updated" in Path(updated_email_path).name)
print()
print("TEST PASSED -- updated publication produced exactly ONE email, correctly rendered as UPDATED with a real comparison.")

# Same updated content, processed again -> NO second email.
r_upd_dup = orchestrator.process_single_publication(
    conn, pub_updated, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(new_text),
)
preview_files_after_upd_dup = set(glob.glob(str(PREVIEW_DIR / "*.html")))
print(f"[re-process updated, unchanged] outcome={r_upd_dup.outcome} status={r_upd_dup.status}")
assert r_upd_dup.outcome == "unchanged" and r_upd_dup.status == "skipped_unchanged"
assert preview_files_after_upd_dup == preview_files_after_upd, "No new email file should have been written."
print("TEST PASSED -- same updated publication processed again produced NO duplicate email.")


## 7. Email provider failure -> scheduler continues

A local `PipelineConfig` copy (never touches the real `config/.env`) points
SMTP at an unreachable local port with `TEST_MODE=false`, so a real SMTP
attempt is genuinely made and genuinely fails fast (connection refused).
The publication must be marked `status='failed'` (never marked as
successfully sent), and `run_dgca_pipeline()` must not raise.


In [ ]:
conn.execute("DELETE FROM publications WHERE publication_id = 'e2e-test-p11-smtp-fail'")
conn.commit()
for f in DOCUMENTS_DIR.glob("e2e-test-p11-smtp-fail*"):
    f.unlink()

broken_smtp_config = PipelineConfig(
    email_provider="smtp",
    test_mode=False,  # deliberately real -- this is what we're testing
    smtp_host="localhost",
    smtp_port=1,  # nothing listens here -- fails fast (connection refused)
    outlook_sender_email="dgca-test-sender@example.invalid",
    outlook_recipient_email="dgca-test-recipient@example.invalid",
    llm_provider="fallback",
)

pub_fail = {
    "source_id": "e2e-test-p11-smtp-fail",
    "title": "Phase 11 Synthetic SMTP-Failure Publication",
    "category": "Regulation",
    "published_date": "2026-08-19",
    "publication_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-smtp-fail",
    "pdf_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-smtp-fail.pdf",
}
r_smtp_fail = orchestrator.process_single_publication(
    conn, pub_fail, broken_smtp_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes("Synthetic content for the SMTP-failure test."),
)
print(f"outcome={r_smtp_fail.outcome} status={r_smtp_fail.status} email_provider={r_smtp_fail.email_provider} email_success={r_smtp_fail.email_success}")
print(f"error={r_smtp_fail.error}")

row_after_fail = conn.execute(
    "SELECT status, email_sent FROM publications WHERE publication_id = 'e2e-test-p11-smtp-fail'"
).fetchone()
print(f"DB row: status={row_after_fail['status']!r} email_sent={row_after_fail['email_sent']}")

assert r_smtp_fail.status == "failed" and not r_smtp_fail.email_success
assert row_after_fail["status"] == "failed" and row_after_fail["email_sent"] == 0, "A failed send must never be marked as sent."
print()
print("TEST PASSED -- real SMTP delivery failure was handled: publication marked 'failed' (never 'sent'), no exception raised.")


## 8. Malformed publication -> isolated failure

Reuses `run_dgca_pipeline()`'s `publications` override (the same controlled-run
parameter Phases 7/10 already built) with valid / malformed / valid records
in one batch.


In [ ]:
conn.execute("DELETE FROM publications WHERE publication_id LIKE 'e2e-test-p11-mal%'")
conn.commit()
for f in DOCUMENTS_DIR.glob("e2e-test-p11-mal*"):
    f.unlink()

batch = [
    {
        "source_id": "e2e-test-p11-mal-a", "title": "Phase 11 Malformed-Batch Valid A", "category": "Announcement",
        "published_date": "2026-08-19", "publication_url": "https://www.dgca.gov.in/digigov-portal/p11-mal-a",
        "pdf_url": "https://www.dgca.gov.in/digigov-portal/p11-mal-a.pdf",
    },
    {
        # Missing required 'title' field -- storage.ingest_publication() rejects this.
        "source_id": "e2e-test-p11-mal-b", "category": "Announcement", "published_date": "2026-08-19",
    },
    {
        "source_id": "e2e-test-p11-mal-c", "title": "Phase 11 Malformed-Batch Valid C", "category": "Announcement",
        "published_date": "2026-08-19", "publication_url": "https://www.dgca.gov.in/digigov-portal/p11-mal-c",
        "pdf_url": "https://www.dgca.gov.in/digigov-portal/p11-mal-c.pdf",
    },
]

# raw_pdf_bytes_override only applies per-call, not per-batch record, so run
# each valid record through process_single_publication directly (same
# function run_dgca_pipeline() calls internally) to keep this deterministic.
results = []
for pub in batch:
    override = make_pdf_bytes(f"Synthetic content for {pub.get('source_id')}.") if "title" in pub else None
    results.append(orchestrator.process_single_publication(
        conn, pub, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False, raw_pdf_bytes_override=override
    ))

for r in results:
    print(f"{r.publication_id}: outcome={r.outcome} status={r.status} error={r.error}")

assert results[0].status == "success"
assert results[1].status == "failed" and results[1].outcome == "failed"
assert results[2].status == "success"
print()
print("TEST PASSED -- malformed record (B) failed in isolation; both valid records (A, C) succeeded.")


## 9. HTML validation

Confirms `validate_email_html()` correctly checks the UPDATED-publication
fields only when `is_update=True` (never for a NEW publication), and
correctly flags absence rather than silently passing.


In [ ]:
# A NEW-publication update must validate cleanly with no UPDATED-only checks.
new_update = email_pipeline.DGCAUpdate(
    title="Validation Test New", category="Guidance", published_date="2026-08-19",
    summary="Summary.", dgca_url="https://dgca.gov.in/x", pdf_url="https://dgca.gov.in/x.pdf",
)
html_new = email_pipeline.render_email_html(new_update, new_update.summary, test_mode=True)
v_new = email_pipeline.validate_email_html(html_new, new_update, new_update.summary)
assert v_new.is_valid
assert not any("change level" in c.lower() or "previous version" in c.lower() for c in v_new.checked), \
    "UPDATED-only checks should not even be attempted for a NEW publication."

# An UPDATED publication with a real comparison must validate cleanly, with
# the comparison fields actually checked and present.
upd_update = email_pipeline.DGCAUpdate(
    title="Validation Test Updated", category="Guidance", published_date="2026-08-19",
    summary="Summary.", dgca_url="https://dgca.gov.in/y", pdf_url="https://dgca.gov.in/y.pdf",
    is_update=True, previous_title="Validation Test Updated (prior version)",
    overall_change_level="LOW", added_changes=["A new clause was added."],
    comparison_explanation="One substantive change detected; review recommended.",
)
html_upd = email_pipeline.render_email_html(upd_update, upd_update.summary, test_mode=True)
v_upd = email_pipeline.validate_email_html(html_upd, upd_update, upd_update.summary)
print(f"Updated-publication validation: is_valid={v_upd.is_valid}, checked={len(v_upd.checked)} items")
assert v_upd.is_valid
assert "Overall change level" in v_upd.checked and "Added change #1" in v_upd.checked

# A deliberately broken UPDATED email (comparison text rendered incorrectly)
# must be caught, not silently passed.
broken_html = html_upd.replace("A new clause was added.", "")
v_broken = email_pipeline.validate_email_html(broken_html, upd_update, upd_update.summary)
print(f"Deliberately-broken email validation: is_valid={v_broken.is_valid}, missing={v_broken.missing}")
assert not v_broken.is_valid and "Added change #1" in v_broken.missing

print()
print("TEST PASSED -- HTML validation correctly gates UPDATED-only checks and correctly detects missing content.")


## 10. Direct PDF link validation -- no invented URLs

The email must use the actual stored `pdf_url`/`dgca_url` verbatim, and must
never fabricate a link when one is genuinely unavailable.


In [ ]:
# Real URL present -> must appear verbatim, and "Download PDF" must link to it.
real_pdf_url = "https://www.dgca.gov.in/digigov-portal/Upload?flag=iframeAttachView&attachId=12345&baseLocale=en_US"
u = email_pipeline.DGCAUpdate(
    title="PDF Link Test", category="Regulation", published_date="2026-08-19",
    summary="Summary.", dgca_url="https://www.dgca.gov.in/digigov-portal/example",
    pdf_url=real_pdf_url,
)
html_with_pdf = email_pipeline.render_email_html(u, u.summary, test_mode=True)
assert real_pdf_url in html_with_pdf, "The actual DGCA PDF URL must appear verbatim -- never rewritten/shortened."
assert "Download PDF" in html_with_pdf and "View on DGCA Website" in html_with_pdf
v1 = email_pipeline.validate_email_html(html_with_pdf, u, u.summary)
assert v1.is_valid

# No PDF URL available -> must NOT invent one; validation must flag it as missing.
u_no_pdf = email_pipeline.DGCAUpdate(
    title="PDF Link Missing Test", category="Regulation", published_date="2026-08-19",
    summary="Summary.", dgca_url="https://www.dgca.gov.in/digigov-portal/example2", pdf_url=None,
)
html_no_pdf = email_pipeline.render_email_html(u_no_pdf, u_no_pdf.summary, test_mode=True)
v2 = email_pipeline.validate_email_html(html_no_pdf, u_no_pdf, u_no_pdf.summary)
print(f"No-PDF-URL case: is_valid={v2.is_valid}, missing={v2.missing}")
assert not v2.is_valid and "PDF link" in v2.missing, "Validation must flag a missing PDF link, not silently pass or invent a URL."
assert 'href="None"' not in html_no_pdf, "Jinja2 would literally render the string 'None' as a URL if we didn't check for this -- confirming it doesn't."

print()
print("TEST PASSED -- real PDF/DGCA URLs render verbatim; a missing URL is correctly flagged, never invented.")


## 11. Explicit (non-automatic) email retry

`storage.mark_for_resend()` (Phase 3) was designed as "the explicit,
operator-triggered escape hatch for a deliberate re-send," but was never
actually wired to anything -- `process_single_publication()` always skipped
on `UNCHANGED` content, so clearing `email_sent` had no effect. Phase 11
completes this wiring with `pipeline.scheduler.retry_failed_notifications()`
-- **not** called automatically by `run_once()`/`run_scheduler()` (so a
permanently broken SMTP config never generates an automatic retry attempt
every single cycle forever), only when explicitly invoked.

This reuses Section 7's synthetic SMTP-failure publication (`status='failed'`,
`email_sent=0` in the real database).


In [ ]:
row_before_retry = conn.execute(
    "SELECT status, email_sent FROM publications WHERE publication_id = 'e2e-test-p11-smtp-fail'"
).fetchone()
print(f"Before retry: status={row_before_retry['status']!r} email_sent={row_before_retry['email_sent']}")
assert row_before_retry["status"] == "failed" and row_before_retry["email_sent"] == 0

# Confirm run_once()/run_dgca_pipeline()'s NORMAL automatic path still does
# NOT retry this on its own (no automatic/indefinite retry).
r_normal_pass = orchestrator.process_single_publication(
    conn, pub_fail, test_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
)
print(f"Normal (non-retry) automatic pass: outcome={r_normal_pass.outcome} status={r_normal_pass.status}")
assert r_normal_pass.status == "skipped_unchanged", "The automatic path must never auto-retry a failed send."

# Now call the explicit retry entry point, with a WORKING config this time
# (the operator fixed the SMTP problem -- here, simply using the safe
# preview provider to demonstrate the retry actually re-sends).
conn.close()  # retry_failed_notifications() opens its own connection
retry_result = scheduler.retry_failed_notifications(config=test_config)
print(f"Retry result: {json.dumps(retry_result, indent=2)}")
assert retry_result["retried"] == 1 and retry_result["processed"] == 1 and retry_result["failed"] == 0

conn = storage.get_connection()
row_after_retry = conn.execute(
    "SELECT status, email_sent FROM publications WHERE publication_id = 'e2e-test-p11-smtp-fail'"
).fetchone()
print(f"After retry: status={row_after_retry['status']!r} email_sent={row_after_retry['email_sent']}")
assert row_after_retry["status"] == "processed" and row_after_retry["email_sent"] == 1

# Calling it again immediately must find nothing left to retry.
retry_result_2 = scheduler.retry_failed_notifications(config=test_config)
print(f"Second retry call: {json.dumps(retry_result_2, indent=2)}")
assert retry_result_2["retried"] == 0

print()
print("TEST PASSED -- retry is explicit (not automatic), successfully resent exactly once, and is a no-op once resolved.")


## 12. Final end-to-end demonstration

A controlled NEW publication through the complete, real chain: Detection ->
PDF processing -> AI analysis (real local Ollama, `LLM_PROVIDER=local`) ->
`DGCAUpdate` -> HTML email -> `EmailSender` -> (preview, since real SMTP
isn't configured yet) -> validated -> database status -> dashboard-ready.
Then the same publication is processed again: expected SKIPPED, no second
email.


In [ ]:
conn.execute("DELETE FROM publications WHERE publication_id = 'e2e-test-p11-final'")
conn.commit()
for f in DOCUMENTS_DIR.glob("e2e-test-p11-final*"):
    f.unlink()

final_config = PipelineConfig(email_provider="preview", test_mode=True, llm_provider="local")
pub_final = {
    "source_id": "e2e-test-p11-final", "title": "Phase 11 Final End-to-End Demonstration Publication",
    "category": "Guidance", "published_date": "2026-08-19",
    "publication_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-final",
    "pdf_url": "https://www.dgca.gov.in/digigov-portal/e2e-test-p11-final.pdf",
}
final_text = (
    "This guidance establishes revised procedures for reporting of bird-strike incidents. "
    "All licensed operators must submit incident reports within forty-eight hours of occurrence. "
    "Non-compliance may result in regulatory action under the applicable Civil Aviation Requirements."
)

preview_before = set(glob.glob(str(PREVIEW_DIR / "*.html")))
r_final = orchestrator.process_single_publication(
    conn, pub_final, final_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(final_text),
)
orchestrator.print_pipeline_result(r_final)
print(f"analysis_mode={r_final.analysis_mode}")

row_final = storage.get_publication_by_id(conn, "e2e-test-p11-final")
print(f"Database status: {row_final['status']!r}, analyzed_at set: {row_final['analyzed_at'] is not None}")

assert r_final.status == "success" and r_final.email_success
assert row_final["status"] == "processed" and row_final["analyzed_at"] is not None

# Run again -> SKIPPED, no second email.
preview_after_first = set(glob.glob(str(PREVIEW_DIR / "*.html")))
r_final_again = orchestrator.process_single_publication(
    conn, pub_final, final_config, DOCUMENTS_DIR, PREVIEW_DIR, ocr_available=False,
    raw_pdf_bytes_override=make_pdf_bytes(final_text),
)
preview_after_second = set(glob.glob(str(PREVIEW_DIR / "*.html")))
print(f"Second run: outcome={r_final_again.outcome} status={r_final_again.status}")
assert r_final_again.status == "skipped_unchanged"
assert preview_after_second == preview_after_first, "No second email should have been generated."

print()
print("FINAL END-TO-END TEST PASSED: new publication -> real local-LLM analysis -> validated email -> ")
print("delivered (preview) -> database status='processed' -> dashboard-ready. Second run: SKIPPED, no duplicate email.")


## 13. Security audit


In [ ]:
import re

gitignore_text = (PROJECT_ROOT / ".gitignore").read_text(encoding="utf-8")
env_ignored = {".env", ".env.*"} <= set(gitignore_text.splitlines())
print(f"Git ignores .env and .env.*: {env_ignored}")

KEY_PATTERNS = [
    re.compile(r"sk-[A-Za-z0-9]{20,}"),          # OpenAI-style key
    re.compile(r"AZURE_CLIENT_SECRET\s*=\s*\S{8,}"),
]
scan_globs = ["notebooks/*.ipynb", "pipeline/*.py", "scheduler/*.py", "dashboard/*.py", "*.md", "logs/*.log"]
files_scanned = 0
findings = []
for pattern in scan_globs:
    for path in (PROJECT_ROOT).glob(pattern):
        if path.name == "11_Real_Email_Notification.ipynb":
            continue  # this notebook is still being written during this very cell
        files_scanned += 1
        text = path.read_text(encoding="utf-8", errors="ignore")
        for kp in KEY_PATTERNS:
            if kp.search(text):
                findings.append(str(path))

print(f"Files scanned: {files_scanned}")
print(f"Files with a real-looking secret pattern found: {len(findings)}")
if findings:
    print(findings)

assert env_ignored
assert len(findings) == 0

print()
print("Security check PASSED -- no API key / client secret pattern found in any notebook, pipeline module, or doc file.")
print("The real SMTP password / OpenAI API key in config/.env were never read or printed by this notebook --")
print("only presence booleans (via pipeline.scheduler.check_email_configuration()) were ever computed.")


## 14. Real delivery test (TEST 3 / TEST 10)

Sends **exactly ONE** clearly-labeled test email
(`[DGCA TEST] Regulatory Intelligence Email Test`) via
`pipeline.email_pipeline.send_test_email()` -- but **only** if real SMTP
configuration is actually present (`EMAIL_PROVIDER=smtp`, `TEST_MODE=false`,
SMTP host/sender/recipient all set). Never attempts a real send otherwise --
this cell is safe to run in any environment.


In [ ]:
live_config = load_config(ENV_PATH)
live_status = scheduler.check_email_configuration(live_config)

ready = (
    live_status["email_provider"] == "smtp" and not live_status["test_mode"]
    and live_status["smtp_configured"] and live_status["recipient_set"]
)

if not ready:
    print("Real delivery SKIPPED -- configuration not yet present for a real send.")
    print(f"  EMAIL_PROVIDER={live_status['email_provider']!r} (need 'smtp')")
    print(f"  TEST_MODE={live_status['test_mode']} (need False)")
    print(f"  SMTP configuration: {'SET' if live_status['smtp_configured'] else 'MISSING'}")
    print(f"  Recipient: {'SET' if live_status['recipient_set'] else 'MISSING'}")
    print()
    print("To send a real test email once config/.env has real SMTP credentials:")
    print('  .venv\\Scripts\\python.exe -c "from pipeline.config import load_config; from pipeline.email_pipeline import send_test_email; '
          'from pathlib import Path; c=load_config(Path(\'config/.env\')); '
          'print(send_test_email(c, Path(\'data/processed/email_previews\')))"')
else:
    result = email_pipeline.send_test_email(live_config, PREVIEW_DIR)
    print(f"Real delivery attempted. success={result.success} provider={result.provider}")
    print(f"message={result.message}")
    assert result.success, "Real delivery was attempted but the provider did not confirm success -- see message above."
    print()
    print("TEST 3/10 PASSED -- provider confirmed successful delivery of exactly one test email.")


## 15. Summary

See the chat response for the full itemized Phase 11 final report. In
short: the existing `EmailSender`/SMTP architecture is now fully wired for
real delivery, gated safely behind `TEST_MODE`; NEW vs UPDATED publications
render distinctly (dynamic subject + version-comparison block, with a
genuine in-place-update comparison capture fix); duplicate prevention,
provider-failure isolation, malformed-record isolation, and an explicit
(non-automatic) retry mechanism were all demonstrated against the real
database. Real delivery itself was only attempted if `config/.env` already
had real SMTP credentials configured -- otherwise it was safely skipped
with exact next-step instructions printed above.
